In [ ]:
# Colab Setup - Run this cell first if using Google Colab
# Skip this cell if running locally with torchref already installed

#install torchref
!pip install torchref

# Download a structure/dataset pair 
!wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.pdb
!wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.mtz

mtzpath = "./1DAW.mtz"
pdbpath = "./1DAW.pdb"

# TorchRef Code Examples

This notebook demonstrates the code examples from the TorchRef README, providing a comprehensive overview of the library's capabilities.

## Setup

First, let's import TorchRef and set up our file paths.

In [2]:
import os
import torchref
import torch
from torchref import ROOT_TORCHREF  # Root directory of torchref installation

# File paths - use Colab-downloaded files if available, else use local paths
if os.path.exists('./1DAW.mtz'):
    # Running in Colab with downloaded files
    mtzpath = './1DAW.mtz'
    pdbpath = './1DAW.pdb'
    directory_notebook = './'
else:
    # Running locally
    directory_notebook = f'{ROOT_TORCHREF}/example_notebooks/'
    mtzpath = f'{directory_notebook}/1DAW.mtz'
    pdbpath = f'{directory_notebook}/1DAW.pdb'

/tmp/ipykernel_2151570/3103064995.py:2: UserWarning: TorchRef auto-configured 4 threads. Set TORCHREF_NUM_THREADS to override.
  import torchref


## Basic functionality

Use a model to compute structure factor and scale them to a dataset

In [3]:
from torchref import ReflectionData
from torchref import ModelFT
from torchref import Scaler

from torchref.base import get_rfactors
# The ReflectionData class can load MTZ files or CIF files
# It by default loads intensities and converts them to structure factors using French-Wilson conversion
# It keeps track of flagged values, rfree flags, and other useful information

data = ReflectionData().load_mtz(mtzpath)

model = ModelFT().load_pdb(pdbpath)

hkl, F, sigF, rfree = data()


# Calculate structure factors for the the model for a given set of hkl
fcalc = model(hkl)

scale = Scaler(model, data)

scale.initialize() # setup initial scale factors, and also bulk solvent if applicable
scale.refine_lbfgs() # refine scale factors using L-BFGS optimizer

scaled_fcalc = scale(fcalc)

Fcalc_abs = torch.abs(scaled_fcalc)

rwork, rfree = get_rfactors(F, Fcalc_abs, rfree)

print(f"Rwork: {rwork:.3f}, Rfree: {rfree:.3f}")


FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
Initialized ScalerBase with 20 bins.
Calculating initial scale factors using 20 bins.
Refining scales with LBFGS...
Scale refinement complete. rwork: 0.2098, rfree: 0.2738

Final Scale Parameters: 
  log_scale: tensor([-6.6710, -6.5879, -6.5617, -6.5045, -6.4921, -6.4635, -6.4246, -6.3889,
        -6.3852, -6.3462, -6.3225, -6.2781, -6.2240, -6.1792, -6.1381, -6.0710,
        -5.9933, -5.9826, -5.9987, -6.0219])
  U: tensor([-0.2655, -0.1760, -0.0912, -0.0035, -0.1583, -0.0029])
  solvent.log_k_solvent: -0.9674057364463806
  solvent.b_solvent: 46.08067321777344
  solvent.phase_offset: -0.0010770070366561413
Rwork: 0.210, Rfree: 0.274


/das/work/p17/p17490/Peter/testing_packages/test_torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


## Model parameters can be selectively frozen

This may be used during refinement or when only parts of the model want to be refined

In [4]:
model = ModelFT().load_pdb(pdbpath)

print([i.shape for i in model.parameters()])

model.freeze('b') # Freeze all B-factor parameters
print([i.shape for i in model.parameters()])

model.unfreeze('b') # Unfreeze all B-factor parameters
print([i.shape for i in model.parameters()])

model.freeze('all') # Freeze all parameters
print([i.shape for i in model.parameters()])

model.unfreeze('all') # Unfreeze all parameters
print([i.shape for i in model.parameters()])

model.freeze('xyz') # Freeze all position parameters

print([i.shape for i in model.parameters()])
model.unfreeze('xyz') # Unfreeze all position parameters

# We can also freeze unfreeze only a part of the model through phenix style selections

selection_str = "chain A and resseq 10:20"

model.freeze_selection(selection_str)

print([i.shape for i in model.parameters()])

# And similarly

model.freeze_all() # First freeze all

model.unfreeze_selection(selection_str) # Then unfreeze selection

print([i.shape for i in model.parameters()])

model.unfreeze_selection("all")  # Unfreeze all again, if we made a selection before we have to do this to unfreeze all

# we can also do this at a parameter level

mask_xyz = torch.zeros(model.xyz.shape, dtype=torch.bool)  # just to illustrate
mask_xyz[1:500,:] = True  # example mask to unfreeze atoms 1 to 500

model.xyz.update_refinable_mask(mask_xyz)

print([i.shape for i in model.parameters()])



Loaded 3051 atoms
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051, 3]), torch.Size([3051]), torch.Size([3])]
[torch.Size([3051]), torch.Size([3])]
Selection 'chain A and resseq 10:20' (91 atoms) frozen for xyz
  Total refinable atoms for xyz: 2960/3051
  Applied mask to xyz: 2960 atoms refinable
Selection 'chain A and resseq 10:20' (91 atoms) frozen for adp
  Total refinable atoms for adp: 2960/3051
  Applied mask to adp: 2960 atoms refinable
Selection 'chain A and resseq 10:20' (91 atoms) frozen for u
  Total refinable atoms for u: 0/3051
  Applied mask to u: 0 atoms refinable
Selection 'chain A and resseq 10:20' (91 atoms) frozen for occupancy
  Total refinable atoms for occupancy: 11/3051
  Applied mask to occupancy: 11 atoms refinable
[torch.Size([2960, 3]), torch.Size([2

## Basic Refinement

### Loading Data and Model Components Separately

In [5]:
from torchref.io import ReflectionData
from torchref.model import ModelFT
from torchref.scaling import Scaler

# Load reflection data
data = ReflectionData(verbose=1)
data.load_mtz(mtzpath)

# Load model with structure factor calculation capability
model = ModelFT()
model.load_pdb(pdbpath)

# Create scaler
scaler = Scaler(model, data)

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 20 bins.


### Initialize Refinement from Files Automatically

In [6]:
from torchref.refinement import LBFGSRefinement

# Initialize refinement directly from files
refinement = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

print(f"Initial R-factors: {refinement.get_rfactor()}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 

/das/work/p17/p17490/Peter/testing_packages/test_torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


### Manual Refinement Initialization

Refinement can also be initialized manually by setting components individually.

In [7]:
from torchref import LBFGSRefinement, ReflectionData, ModelFT, Scaler

# Create components
data_manual = ReflectionData()
data_manual.load_mtz(mtzpath)

model_manual = ModelFT()
model_manual.load_pdb(pdbpath)
model_manual._build_restraints() # manually build restraints, normally initialized lazily on first access. 


scaler_manual = Scaler(model_manual, data_manual)


# Manual initialization
refinement_manual = LBFGSRefinement()
refinement_manual.model = model_manual
refinement_manual.reflection_data = data_manual
refinement_manual.scaler = scaler_manual
# Initialize standard targets
refinement_manual._init_targets()

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 0
  Disulfide angles: 0
  Disulfide t

### Running Refinement

In [8]:
from torchref.refinement import LBFGSRefinement

mtzpath = f'./1DAW.mtz'
pdbpath = f'./1DAW.pdb'

# Initialize refinement directly from files
refinement = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)
# Shake coordinates to simulate a starting model with errors
refinement.model.shake_coords(0.1)

loss_state = refinement.refine(macro_cycles=5)

print(f"Final R-factors: {refinement.get_rfactor()}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 

/das/work/p17/p17490/Peter/testing_packages/test_torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


### Writing Output Files

In [9]:
# Write refined structure
refinement.write_out_pdb(f"{directory_notebook}/refined.pdb")

# Write structure factors
refinement.write_out_mtz(f"{directory_notebook}/refined.mtz")

Added map coefficients:
  2mFo-DFc: FWT, PHWT
  mFo-DFc: DELFWT, PHDELWT
  Resolution range: 2.05 - 69.56 Å
✓ Wrote MTZ file: .//refined.mtz
  Reflections: 23356
  Columns: H, K, L, F-obs, SIGF-obs, I-obs, SIGI-obs, R-free-flags, 2FOFCWT, PH2FOFCWT, FOFCWT, PHFOFCWT, F-model, PH-model


## Custom Target Functions

One of TorchRef's key strengths is the ease of defining custom refinement targets. Thanks to PyTorch's automatic differentiation, you simply define the forward computation.

In [10]:
import torch
from torchref.refinement import ModelTarget, DataTarget

class CustomTargetData(DataTarget):
    """Custom refinement target with automatic gradient computation."""
    name = 'test_target_data'

    def forward(self):
        # Define your target function - gradients computed automatically!
        hkl, F_obs, sig_F_obs, rfree = self.data()
        F_calc = self.get_F_calc_scaled()

        # Custom loss computation (simple least squares example)
        loss = torch.mean((torch.abs(F_calc) - F_obs) ** 2)
        return loss
    
class CustomTargetGeom(ModelTarget):
    """Custom refinement target with automatic gradient computation."""

    def forward(self):
        # Define your target function - gradients computed automatically!

        deviations, sigma = self.model.restraints.bond_deviations()
        # Custom loss computation (simple least squares example)

        scaled_deviations = deviations **2 / sigma **2 
        return torch.sum(scaled_deviations)


### Registering Custom Targets

Custom targets are registered in the LossState of the refinement. This breaks the circular dependency and allows easy integration with weighting schemes.

In [11]:
# Create a fresh refinement for this example
refinement_custom = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Create loss state
loss_state = refinement_custom.create_loss_state()

test_target = CustomTargetData(data=refinement_custom.data, model=refinement_custom.model, scaler=refinement_custom.scaler)
name = test_target.name
# Register the custom target
loss_state.register_target(name, test_target)

# Set the weight of the custom target
loss_state.set_weight(name, 3.0)

print(f"Registered targets: {list(loss_state.targets.keys())}")
print(f"Weights: {loss_state.weights}")

# Create loss state
loss_state = refinement_custom.create_loss_state()

test_target = CustomTargetGeom(model=refinement_custom.model)
name = test_target.name
# Register the custom target
loss_state.register_target(name, test_target)

# Set the weight of the custom target
loss_state.set_weight(name, 3.0)

print(f"Registered targets: {list(loss_state.targets.keys())}")
print(f"Weights: {loss_state.weights}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 

## GPU Acceleration

Structure factor calculations parallelize naturally on GPUs. Move everything to GPU for significant speedups.

In [12]:
# Check if CUDA is available
if torch.cuda.is_available():
    print(f"CUDA available: {torch.cuda.get_device_name(0)}")
    
    # Create refinement on GPU
    refinement_gpu = LBFGSRefinement(
        data_file=mtzpath,
        pdb=pdbpath,
    )
    
    # Move to GPU
    refinement_gpu.cuda()
    
    # Create loss state and move to GPU
    loss_state_gpu = refinement_gpu.create_loss_state()
    loss_state_gpu.cuda()
    
    print(f"Model device: {refinement_gpu.model.xyz.device}")
    print(f"LossState device: {loss_state_gpu.device}")
else:
    print("CUDA not available, running on CPU")

CUDA available: Tesla T4
FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles

## Saving and Loading State

TorchRef supports full `state_dict` for saving and loading complete refinement states.

In [13]:
import torch
from torchref.refinement import LBFGSRefinement

# Create refinement
refinement_save = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Save complete refinement state
# torch.save(refinement_save.state_dict(), "checkpoint.pt")

# Load state into new refinement object
# new_refinement = LBFGSRefinement()
# new_refinement.load_state_dict(torch.load("checkpoint.pt"))

print("State dict keys:", list(refinement_save.state_dict().keys())[:10], "...")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 

## Integration with Machine Learning

TorchRef's PyTorch foundation enables seamless integration with neural networks. Here's an example of a hybrid model combining crystallographic refinement with a neural network.

In [ ]:
# Setup simple forward pass function for calculating structure factors

from torchref.base import (
    get_real_grid,                          # Generate fractional coordinate grid
    vectorized_add_to_map,                  # Add isotropic atom contributions
    vectorized_add_to_map_aniso,            # Add anisotropic atom contributions  
    find_relevant_voxels,                   # Find voxels within atom's influence radius
    ifft,                                   # Inverse FFT (real -> reciprocal space)
    extract_structure_factors_with_symmetry # Sample F at HKL with symmetry
)

def compute_structure_factors(hkl, cell, spacegroup, parameters):

    gridsize = cell.compute_grid_size(max_res=2.0)

    # Create fractional coordinate grid (values 0 to 1 in each dimension)
    fractional_grid = get_real_grid(
        fractional_matrix=cell.fractional_matrix, 
        gridsize=gridsize
    )

    # Initialize empty density map
    Electron_density_map = torch.zeros(gridsize, dtype=dtypes.float)

    # Step 2: Unpack atomic parameters
    xyz_iso, adp_iso, occ_iso, A_iso, B_iso = parameters

    # Step 3: Build isotropic density contributions

    surrounding_coords, voxel_indices = find_relevant_voxels(
        fractional_grid,
        xyz_iso,
        radius_angstrom=3.0,  # Cutoff radius - larger = more accurate but slower
        inv_frac_matrix=cell.inv_fractional_matrix,
    )

    # Add Gaussian density contributions from all isotropic atoms
    # Uses the 4-Gaussian approximation to atomic scattering factors
    density_map = vectorized_add_to_map(
        surrounding_coords,    # Voxel positions near each atom
        voxel_indices,         # Which voxels to update
        Electron_density_map,  # Map to add to (modified in place)
        xyz_iso,               # Atom positions (fractional)
        adp_iso,               # B-factors
        cell.inv_fractional_matrix,
        cell.fractional_matrix,
        A_iso,                 # Scattering factor A coefficients
        B_iso,                 # Scattering factor B coefficients
        occ_iso,               # Occupancies
    )

    # Step 5: FFT to reciprocal space
    reciprocal_grid = ifft(density_map)

    # Step 6: Extract structure factors at HKL positions
    f_calc_manual = extract_structure_factors_with_symmetry(
        reciprocal_grid,
        hkl,
        spacegroup.matrices,      # Rotation matrices for symops
        spacegroup.translations   # Translation vectors for symops
    )

    return f_calc_manual

In [15]:
import torch
import torch.nn as nn
from torchref import LBFGSRefinement
from torchref.model import FFT

dataset = ReflectionData()
dataset.load_mtz(mtzpath)

# You can get cell, spacegroup and hkl from the dataset object
spacegroup = dataset.spacegroup
cell = dataset.cell
hkl = dataset.hkl

class HybridModel(nn.Module):
    """Combine crystallographic refinement with a neural network."""
    
    def __init__(self):
        super().__init__()
        self.neural_prior = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 4)  # Predicts coordinates and adp
        )
        self.SF_calculation = compute_structure_factors

    def forward(self, features, cell, spacegroup, hkl):
        # Neural network predictions
        parameters = self.neural_prior(features)

        return self.SF_calculation(hkl,cell, spacegroup, parameters)

        
# This is just a demonstration of how something like this could be done.
# In practice we would also need some kind of alignment which is WIP. 
# Also you would have to implement a basic scaling step. Have a look at the Scaler class for inspiration.

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


## Automatic Differentiation Advantage

Traditional refinement programs require explicit implementation of gradients. TorchRef eliminates this burden using PyTorch's autograd.

In [16]:
# Demonstrate with actual refinement
demo_refinement = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Compute loss - parameters already have requires_grad=True
loss_state = demo_refinement.create_loss_state()
demo_refinement.add_target_info_to_state(loss_state)
demo_refinement.populate_state_meta(loss_state)
demo_refinement.update_weights(loss_state)

loss = loss_state.aggregate()
print(f"Loss: {loss.item():.4f}")

# Gradients computed automatically via PyTorch autograd!
loss.backward()

# Access gradients through refinement.parameters()
params = list(demo_refinement.parameters())
grads_computed = sum(1 for p in params if p.grad is not None)
total_grad_norm = sum(p.grad.norm().item() for p in params if p.grad is not None)
print(f"Parameters with gradients: {grads_computed}/{len(params)}")
print(f"Total gradient norm: {total_grad_norm:.4f}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 

## Loss State Management

The LossState object is central to TorchRef's refinement workflow. It manages targets, weights, and metadata.

In [17]:
# Create refinement
ref = LBFGSRefinement(
    data_file=mtzpath,
    pdb=pdbpath,
)

# Create the loss state for refinement
loss_state = ref.create_loss_state()

# Add target info to the loss state needed for updating weights
ref.add_target_info_to_state(loss_state)

# Populate the state with meta info also needed for weights
ref.populate_state_meta(loss_state)

# Update/set weights in the loss state
ref.update_weights(loss_state)

loss = loss_state.aggregate()

loss.backward()

print(ref.model.xyz.refinable_params.grad)
print(ref.model.adp.refinable_params.grad)
print(ref.model.occupancy.refinable_params.grad)


FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Initialized ScalerBase with 10 bins.
Building restraints...
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Bonds: 2511
  Angles: 2850
  Torsions: 1074
  Planes: 825
  Chirals: 407

INTER-RESIDUE RESTRAINTS:
--------------------------------------------------------------------------------
  Peptide bonds: 326
  Peptide angles: 978
  Disulfide bonds: 

## Summary

This notebook demonstrated:

1. **Basic Refinement** - Loading data, models, and running refinement
2. **Custom Targets** - Defining and registering custom loss functions
3. **GPU Acceleration** - Moving computations to CUDA devices
4. **State Management** - Saving and loading refinement checkpoints
5. **ML Integration** - Combining neural networks with crystallographic refinement
6. **Automatic Differentiation** - PyTorch's autograd for gradient computation
7. **LossState** - Managing targets, weights, and metadata

For more examples, see the `examples/` directory in the TorchRef repository.